<a href="https://colab.research.google.com/github/aayush8644/data-structure-and-communicationlab/blob/main/lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
analyzer = CorrelationAnalyzer(df['hours_studied'], df['Exam_Score'])
manual_results = analyzer.get_all_stats()
slope, intercept = analyzer.regression_coefficients()

# libaray analysis
pearson_lib, p_value_lib = stats.pearsonr(df['hours_studied'], df['Exam_Score'])
r_squared_lib = pearson_lib**2

n = len(df)
k = 1
adj_r_squared_lib = 1 - ((1 - pearson_lib**2) * (n - 1) / (n - k - 1))

In [7]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import math

class CorrelationAnalyzer:

  def __init__(self, x, y):
        self.x = np.array(x)
        self.y = np.array(y)
        self.n = len(x)

  def mean(self,data):
      return sum(data)/len(data)

  def pearson_correlation(self):
      x_mean=self.mean(self.x)
      y_mean=self.mean(self.y)

      sum_xy=sum(self.x[i]*self.y[i] for i in range(self.n))
      sum_x=sum(self.x)
      sum_y=sum(self.y)
      sum_x2=sum(x**2 for x in self.x)
      sum_y2=sum(v**2 for v in self.y)

      numerator=self.n*sum_xy-sum_x*sum_y
      denominator=math.sqrt((self.n*sum_x2-sum_x**2)*
                            (self.n*sum_y2-sum_y**2))

      return numerator/denominator if denominator != 0 else 0


  def coefficient_determination(self):
    return self.pearson_correlation()**2

  def adjusted_r_squared(self):
    r_sq=self.coefficient_determination()
    # For simple linear regression, k=1 (number of predictors). The formula is 1 - (1-R^2)*(n-1)/(n-k-1)
    # Here k is 1, so n-k-1 becomes n-2
    return 1-(1-r_sq)*(self.n-1)/(self.n-2)

  def t_statistic(self):
    r=self.pearson_correlation()
    # Handle case where r is close to 1 or -1 to avoid division by zero or sqrt of negative
    if abs(r) == 1:
        return float('inf') if r > 0 else float('-inf')
    return r*math.sqrt((self.n-2)/(1-r**2))

  def p_value(self):
    t_stat=self.t_statistic()
    df_t_test=self.n-2 # Renamed df to df_t_test to avoid conflict with the DataFrame 'df'
    if df_t_test <= 0:
        return 1.0 # Or raise an error for insufficient data
    return 2*(1-stats.t.cdf(abs(t_stat),df_t_test))

  def regression_coefficients(self):
    x_mean=self.mean(self.x)
    y_mean=self.mean(self.y)

  #calculate slope
    numerator=sum((self.x[i]-x_mean)*(self.y[i]-y_mean) for i in range(self.n))
    denominator=sum((x_val-x_mean)**2 for x_val in self.x)
    slope = numerator/denominator if denominator != 0 else 0

  #calculate intercept
    intercept=y_mean-slope*x_mean
    return slope,intercept


  def get_all_stats(self):
    r=self.pearson_correlation()
    return{
        'n':self.n,
        'r':r,
        'R_squared':r**2,
        'adjusted_R_squared':self.adjusted_r_squared(),
        't_statistic':self.t_statistic(),
        'p_value':self.p_value(),
        'degrees_of_freedom':self.n-2

    }

# Define DataFrame
data = {
    'hours_studied': [1, 2, 2.5, 3, 4, 4.5, 5, 6, 7, 8, 8.5, 9, 10],
    'Exam_Score': [55, 60, 65, 68, 72, 80, 85, 88, 89, 90, 92, 98, 99]
}
df = pd.DataFrame(data)

# Ensure necessary variables are defined
analyzer = CorrelationAnalyzer(df['hours_studied'], df['Exam_Score'])
manual_results = analyzer.get_all_stats()
slope, intercept = analyzer.regression_coefficients()

# libaray analysis
pearson_lib, p_value_lib = stats.pearsonr(df['hours_studied'], df['Exam_Score'])
r_squared_lib = pearson_lib**2

n = len(df)
k = 1
adj_r_squared_lib = 1 - ((1 - pearson_lib**2) * (n - 1) / (n - k - 1))

#results output
print("Manual Implementation(From Scratch):")
for key, value in manual_results.items():
    if key == 'p_value':
        print(f"{key:25}: {value:.6e}")
    else:
        print(f"{key:25}: {value:.6f}")


print("\nREGRESSION LINE")
print(f"Slope (m): {slope:.6f}")
print(f"Intercept (b): {intercept:.6f}")

print(f"Equation    :Exam_Score = {intercept:.2f} + {slope:.2f} * Hours_Studied")

print("\nLIBRARY BASED VALIDATION")
print(f"Pearson r (Scipy) : {pearson_lib:.6f}")
print(f"R-squared : {r_squared_lib:.6f}")
print(f"Adjusted R-squared  : {adj_r_squared_lib:.6f}")

print("\nHYPOTHESIS TESTING")
print(f"Null Hypothesis (H0) : p = 0 (No correlation)")
print(f"Alternative (H1) : p ≠ 0 (Significant correlation)")

print(f"t-statistic : {manual_results['t_statistic']:.4f}")
print(f"Degrees of Freedom : {manual_results['degrees_of_freedom']}")
print(f"p-value : {manual_results['p_value']:.6e}")
print(f"Significance Level(@) : 0.05")
print(f"Decision:{'reject h0'if manual_results['p_value']<0.05 else'fail to reject h0'}")
print(f"conclusion:{'signficant correctilanl exists' if manual_results['p_value']<0.05 else 'no significant correlation'}")

Manual Implementation(From Scratch):
n                        : 13.000000
r                        : 0.970214
R_squared                : 0.941314
adjusted_R_squared       : 0.935979
t_statistic              : 13.283051
p_value                  : 4.068006e-08
degrees_of_freedom       : 11.000000

REGRESSION LINE
Slope (m): 4.871445
Intercept (b): 53.658703
Equation    :Exam_Score = 53.66 + 4.87 * Hours_Studied

LIBRARY BASED VALIDATION
Pearson r (Scipy) : 0.970214
R-squared : 0.941314
Adjusted R-squared  : 0.935979

HYPOTHESIS TESTING
Null Hypothesis (H0) : p = 0 (No correlation)
Alternative (H1) : p ≠ 0 (Significant correlation)
t-statistic : 13.2831
Degrees of Freedom : 11
p-value : 4.068006e-08
Significance Level(@) : 0.05
Decision:reject h0
conclusion:signficant correctilanl exists
